# LangGraph third-party agent

This notebook is a **LangGraph ReAct agent** calling CDP Agent Gateway the way a partner SDK would: POST JSON-RPC to `/mcp/spark`, `/mcp/hive`, and `/mcp/impala` with a Knox JWT. It never talks to Knox, Livy, HiveServer2, or Impala directly, and it does not use Streamable HTTP.

The hardcoded Spark → Hive walkthrough stays in [`third_party_agent.ipynb`](third_party_agent.ipynb). This notebook tests the agent-framework path: discover tools, bind them, let the graph call them.

Cloudera AI Workbench already has **LangChain 0.3**. The install cell pins `langchain-core>=0.3.85,<0.4` and LangGraph 0.3. It must not install langchain-core 1.x (that breaks `langchain`, `langchain-aws`, and `langchain-community` on the runtime). If a previous cell installed 1.x, restart the session and re-run.

**Session secrets (not git, not AMP project env):**

- Knox JWT — paste in the token cell (`getpass`). Optional: `KNOX_TOKEN` for this engine.
- Model key — `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` for this engine. Optional: `LANGGRAPH_MODEL`.

Do not print either secret. Compose MCP also sends `X-Agent-Key` (default `lab-agent`).

In [ ]:
import os
import sys
from pathlib import Path

root = Path(os.environ.get("AGENTGATEWAY_ROOT") or Path.cwd())
if not (root / "pyproject.toml").is_file():
    alt = Path("/home/cdsw")
    if (alt / "pyproject.toml").is_file():
        root = alt
agent_dir = root / "examples" / "agent"
src = root / "src"
for path in (agent_dir, src):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from langgraph_mcp import (
    chat_model,
    install_langgraph_deps,
    invoke_agent,
    langchain_tools,
    last_ai_text,
    make_agent,
    tool_names_used,
)
from mcp_agent import load_knox_token, mcp_base_url, profile

install_langgraph_deps(root=root)

print("profile:", profile())
print("spark url:", mcp_base_url("spark"))
print("hive url:", mcp_base_url("hive"))
print("impala url:", mcp_base_url("impala"))

## Knox JWT (this session only)

AMP project env is for `KNOX_PROXY_URL`, not the user bearer. Paste a Knox Token API JWT below. It is stored in `os.environ` for this engine only and is never printed.

In [ ]:
token = load_knox_token(prompt=True)
print("knox jwt: set" if token else "knox jwt: missing")

## Bind MCP tools

`tools/list` on each adapter becomes LangChain tools. The ReAct graph then calls `tools/call` with the same Knox JWT. This is not `langchain-mcp-adapters` Streamable HTTP.

In [ ]:
tools = langchain_tools()
print("bound:", ", ".join(sorted(t.name for t in tools)))
agent = make_agent(chat_model(), tools=tools)
print("graph:", type(agent).__name__)

## Read-only agent turn

Ask for catalogs the Knox subject can see. This is the default third-party test: no `spark_submit_batch`, so it is safe against a live cluster and finishes quickly.

Override the prompt with `LANGGRAPH_QUESTION`.

In [ ]:
question = (os.environ.get("LANGGRAPH_QUESTION") or "").strip() or (
    "List Spark MCP batches for this Knox user, then list Hive databases Ranger allows. "
    "Summarize tool names you used. Do not submit a Spark job."
)
print("question:", question)
result = invoke_agent(question, agent=agent)
print("tools used:", ", ".join(tool_names_used(result)) or "(none)")
print(last_ai_text(result))

## Optional write path

Set `LANGGRAPH_RUN_SUBMIT=1` only after the job file is on a Ranger-allowed URI (`gateway webhdfs put` on Compose, or `SPARK_FILE_URI` on AMP). The agent may call `spark_submit_batch` (a write as the Knox subject) and poll `spark_get_batch`. That can take several minutes and counts against MCP burst + daily submit quota.

For a deterministic submit → Hive select, use [`third_party_agent.ipynb`](third_party_agent.ipynb) instead.

In [ ]:
if os.environ.get("LANGGRAPH_RUN_SUBMIT", "").strip() not in {"1", "true", "yes"}:
    print("skip submit (set LANGGRAPH_RUN_SUBMIT=1 to enable)")
else:
    from mcp_agent import knox_user_from_spark, spark_job_uri

    knox_user = knox_user_from_spark()
    file_uri = spark_job_uri(knox_user)
    database = knox_user.split("@", 1)[0]
    submit_question = (
        f"Submit {file_uri} with spark_submit_batch name count-to-10. "
        f"Poll spark_get_batch until success or dead. "
        f"If success, hive_select database={database} table=count_to_10 columns n limit 10. "
        "Do not invent SQL."
    )
    print("knox_user:", knox_user)
    print("file:", file_uri)
    submit_result = invoke_agent(submit_question, agent=agent, recursion_limit=50)
    print("tools used:", ", ".join(tool_names_used(submit_result)) or "(none)")
    print(last_ai_text(submit_result))